## Auskunft

In [1]:
import logging

class Database:
  def __init__(self):
    self.logger = logging.getLogger('vs2lab.lab1.telefonauskunft_server.Database') # setup logger
    self.phones = {'Anna Mueller': '12345678', 'Hans Schmidt': '87654321'} # phone database
    self.logger.info('phone database initialized')
  
  def process_request(self, request, **kwargs):
    if request == 'GET':
      self.logger.info('calling GET...')
      return self.GET(kwargs.get('name', '')) # call GETALL method
    elif request == 'GETALL':
      self.logger.info('calling GETALL')
      return self.GETALL() # call GETALL method
    else:
      self.logger.error('command not found') # unknown command
  
  def GET(self, name):
    if name: # name in database - return phone number
      for key in self.phones.keys():
        if key == name:
          return str({key: self.phones[key]})
    else: # name not in database - return error
      return 'no entry found'

  def GETALL(self):
    return str(self.phones)

## Transport

In [2]:
import clientserver
import logging
import socket

class Transport:
  def __init__(self):
    self.logger = logging.getLogger('vs2lab.lab1.telefonauskunft_server.Transport') # setup logger
    self.logger.info('logger initialized')

    self.database = Database() # setup database
    self.serving = True
    self.sock = clientserver.Server().sock
    self.serve() # start serving

  def serve(self):
    self.sock.listen(1) # allow one pending connection
    while self.serving:
      self.logger.info('waiting for connection...')
      try:
        connection, address = self.sock.accept() # try establishing connection
        self.logger.info(f'connection established with address {address}')
        receiving = True
        while receiving:
          data = connection.recv(1024) # receive one kilobyte of data at a time
          if not data:
            self.logger.info('no data receiving, closing connection...')
            receiving = False
            continue
          decoded = data.decode('ascii')
          parts = decoded.split(':')
          print(parts)
          response = self.database.process_request(parts[0], name=parts[1])
          connection.send(response.encode('ascii'))
        self.logger.info('response sent, closing connection...')
        connection.close() 
        self.logger.info('connection closed')
      except socket.timeout:
        self.logger.error('connection timeout')
    self.sock.close() # close TCP connection
    self.logger.info('shuting down server...')

## Example

In [3]:
server = Transport()

2025-11-05 16:39:50,972 - vs2lab.lab1.telefonauskunft_server.Transport - INFO - logger initialized
2025-11-05 16:39:50,977 - vs2lab.lab1.telefonauskunft_server.Database - INFO - phone database initialized
2025-11-05 16:39:50,980 - vs2lab.lab1.clientserver.Server - INFO - Server bound to socket <socket.socket fd=79, family=2, type=1, proto=0, laddr=('127.0.0.1', 50007)>
2025-11-05 16:39:50,981 - vs2lab.lab1.telefonauskunft_server.Transport - INFO - waiting for connection...
2025-11-05 16:39:53,986 - vs2lab.lab1.telefonauskunft_server.Transport - ERROR - connection timeout
2025-11-05 16:39:53,987 - vs2lab.lab1.telefonauskunft_server.Transport - INFO - waiting for connection...
2025-11-05 16:39:56,992 - vs2lab.lab1.telefonauskunft_server.Transport - ERROR - connection timeout
2025-11-05 16:39:56,993 - vs2lab.lab1.telefonauskunft_server.Transport - INFO - waiting for connection...
2025-11-05 16:39:59,107 - vs2lab.lab1.telefonauskunft_server.Transport - INFO - connection established with ad

['GET', 'Anna Mueller']
['GETALL', '']


2025-11-05 16:40:02,148 - vs2lab.lab1.telefonauskunft_server.Transport - ERROR - connection timeout
2025-11-05 16:40:02,149 - vs2lab.lab1.telefonauskunft_server.Transport - INFO - waiting for connection...
2025-11-05 16:40:05,154 - vs2lab.lab1.telefonauskunft_server.Transport - ERROR - connection timeout
2025-11-05 16:40:05,156 - vs2lab.lab1.telefonauskunft_server.Transport - INFO - waiting for connection...
2025-11-05 16:40:08,160 - vs2lab.lab1.telefonauskunft_server.Transport - ERROR - connection timeout
2025-11-05 16:40:08,162 - vs2lab.lab1.telefonauskunft_server.Transport - INFO - waiting for connection...


KeyboardInterrupt: 

## Tests

In [ ]:
import unittest

class ServerTest(unittest.TestCase):
  def setUp(self):
    self.db = Database()

  def test_should_contain_all_entries(self):
    result = self.db.process_request('GETALL')
    self.assertIn('Anna Mueller', result)
    self.assertIn('12345678', result)
    self.assertIn('Hans Schmidt', result)
    self.assertIn('87654321', result)
  
  def test_should_contain_entry(self):
    result = self.db.process_request('GET', name='Anna Mueller')
    self.assertIn('Anna Mueller', result)
    self.assertIn('12345678', result)
  
  def test_should_fail_with_no_name(self):
    result = self.db.process_request('GET', name='')
    self.assertEqual(result, 'no entry found')

if __name__ == '__main__':
  unittest.main(argv=['ignore-first-arg'], exit=False) # avoid sys.argv args and exiting


...
----------------------------------------------------------------------
Ran 3 tests in 0.002s

OK
